# 02_Exploratory_Data_Analysis

This notebook is dedicated **only** to Exploratory Data Analysis (EDA) for the cleaned food delivery dataset.

- Dataset used: `food_delivery_clean.csv`
- No data cleaning is performed here
- No rows are removed
- No values are modified
- No new features are created
- All analysis is read-only and business-focused

In [ ]:
import warnings\n
warnings.filterwarnings('ignore')\n
\n
import numpy as np\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
import seaborn as sns\n
import plotly.express as px\n
from IPython.display import display, Markdown\n
\n
sns.set_theme(style='whitegrid', context='talk')\n
plt.rcParams['figure.figsize'] = (14, 7)\n
plt.rcParams['axes.titlesize'] = 16\n
plt.rcParams['axes.labelsize'] = 12\n
\n
def story(title, observation, insight, recommendation):\n
    text = f"""### 📊 {title}\n
\n
🔍 **Observation**  \n
{observation}\n
\n
💡 **Business Insight**  \n
{insight}\n
\n
🚀 **Recommendation**  \n
{recommendation}\n
"""\n
    display(Markdown(text))\n

In [ ]:
df = pd.read_csv('food_delivery_clean.csv')

month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

print('Dataset loaded successfully for EDA.')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
print('Analysis will use the cleaned dataset exactly as provided.')

# Dataset Overview

In [ ]:
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

print('\nColumns:')
print(list(df.columns))

print('\nData Types:')
print(df.dtypes)

print(f'\nDuplicate rows: {df.duplicated().sum():,}')

missing_summary = df.isna().sum().sort_values(ascending=False)
print('\nTop columns with missing values:')
print(missing_summary[missing_summary > 0].head(15))

print('\nSummary Statistics:')
display(df.describe(include='all').T)

print('\nSample Records:')
display(df.head(5))

# 1. Univariate Analysis

In [ ]:
num_cols = [\n
    'Total_Amount', 'Profit', 'Delivery_Time_Min', 'Distance_km', 'Customer_Rating',\n
    'Restaurant_Rating', 'Items_Count', 'Average_Order_Value', 'Profit_Margin_%'\n
]\n
\n
fig, axes = plt.subplots(3, 3, figsize=(22, 16))\n
for i, col in enumerate(num_cols):\n
    r, c = divmod(i, 3)\n
    sns.histplot(df[col].dropna(), kde=True, ax=axes[r, c], color='#2A9D8F', bins=30)\n
    axes[r, c].set_title(f'Distribution of {col}')\n
plt.tight_layout()\n
plt.show()\n
\n
fig, axes = plt.subplots(3, 3, figsize=(22, 16))\n
for i, col in enumerate(num_cols):\n
    r, c = divmod(i, 3)\n
    sns.boxplot(y=df[col], ax=axes[r, c], color='#E9C46A')\n
    axes[r, c].set_title(f'Box Plot of {col}')\n
plt.tight_layout()\n
plt.show()\n
\n
fig, axes = plt.subplots(3, 3, figsize=(22, 16))\n
for i, col in enumerate(num_cols):\n
    r, c = divmod(i, 3)\n
    sns.violinplot(y=df[col], ax=axes[r, c], color='#264653')\n
    axes[r, c].set_title(f'Violin Plot of {col}')\n
plt.tight_layout()\n
plt.show()\n
\n
story(\n
    'Numerical Variable Distributions and Outliers',\n
    f"Total revenue ranges from {df['Total_Amount'].min():.2f} to {df['Total_Amount'].max():.2f} with median {df['Total_Amount'].median():.2f}. Profit ranges from {df['Profit'].min():.2f} to {df['Profit'].max():.2f}, and delivery time spans {df['Delivery_Time_Min'].min()} to {df['Delivery_Time_Min'].max()} minutes. Box and violin plots show right-skew and several high-value outliers in value and profit metrics.",\n
    'Order economics are healthy but uneven: a smaller set of high-value orders contributes disproportionately to topline and profit, while service times vary significantly across the order base.',\n
    'Protect and grow high-value baskets through premium bundles and targeted upsell, and separately run SLA improvement programs for long-tail high-delivery-time cohorts.'\n
)

## 📊 Numerical Distributions and Outliers

### 🔍 Observation
- Total revenue is ₹4,307,370.43 and total profit is ₹969,267.77, but both are right-skewed rather than evenly distributed.
- Delivery time averages 45.05 minutes with a median of 45.00, while profit margin sits at 22.49% with an upper quartile of 26.33%.
- The high-delay bucket averages 56.9 minutes, which shows that the slowest orders are materially longer than the typical order flow.

### 💡 Business Insight
- A relatively small share of orders is driving a disproportionate share of topline and profit.
- The wide spread in delivery and margin values means service and economics need separate monitoring, not one blended KPI.

### 🚀 Recommendation
- Track premium orders and high-delay orders in separate daily alerts for city operations.
- Use the top quartile margin band to identify which order types can absorb targeted upsell or delivery promises.

In [ ]:
cat_cols = [\n
    'City', 'Cuisine', 'Customer_Segment', 'Payment_Method', 'Order_Status',\n
    'Traffic_Level', 'Weather', 'Delivery_Type', 'Festival', 'Peak_Hour', 'Weekend'\n
]\n
\n
fig, axes = plt.subplots(4, 3, figsize=(24, 20))\n
axes = axes.flatten()\n
for i, col in enumerate(cat_cols):\n
    order = df[col].value_counts().index\n
    sns.countplot(data=df, x=col, order=order, ax=axes[i], palette='viridis')\n
    axes[i].set_title(f'Count of {col}')\n
    axes[i].tick_params(axis='x', rotation=30)\n
for j in range(len(cat_cols), len(axes)):\n
    axes[j].axis('off')\n
plt.tight_layout()\n
plt.show()\n
\n
status_share = df['Order_Status'].value_counts(normalize=True).mul(100)\n
fig, axes = plt.subplots(1, 2, figsize=(16, 6))\n
df['Order_Status'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0], colors=sns.color_palette('Set2'))\n
axes[0].set_ylabel('')\n
axes[0].set_title('Order Status Distribution')\n
df['Payment_Method'].value_counts().plot(kind='bar', ax=axes[1], color='#457B9D')\n
axes[1].set_title('Payment Method Mix')\n
axes[1].tick_params(axis='x', rotation=30)\n
plt.tight_layout()\n
plt.show()\n
\n
story(\n
    'Categorical Mix: Demand, Fulfillment, and Payment Behavior',\n
    f"Delivered orders account for {status_share.get('Delivered', 0):.1f}% of total volume, while cancellations/refunds make up the remainder. Category charts show concentration in selected cities, cuisines, and payment methods rather than uniform demand.",\n
    'Demand concentration means growth and risk both sit in a few high-impact segments. Payment and fulfillment mix also indicates where operational efficiency and customer trust can be improved fastest.',\n
    'Prioritize top-contributing city-cuisine combinations, improve reliability in refund/cancel cohorts, and align payment incentives with preferred channels for higher conversion.'\n
)

## 📊 Categorical Mix: Demand, Fulfillment, and Payment Behavior

### 🔍 Observation
- Delivered orders account for 90.1% of volume, while 5.9% were cancelled and 4.0% were refunded.
- Cash generated ₹965,587.77 in revenue (22.4% of total), followed by UPI at ₹861,091.18 (20.0%).
- Afternoon is the busiest order window with 1,596 orders, and weekends contributed ₹1,261,620.06 in revenue.

### 💡 Business Insight
- Fulfillment reliability is strong overall, but the 9.9% non-delivered share is still big enough to affect trust and repeat behavior.
- Payment and timing patterns show where conversion, settlement speed, and surge handling matter most.

### 🚀 Recommendation
- Promote UPI cashback and checkout nudges in high-volume cities to shift part of cash-heavy demand into faster digital flows.
- Increase rider and support coverage in the afternoon and on weekends, where demand concentration is highest.

# 2. Bivariate Analysis

In [ ]:
rev_city = df.groupby('City', as_index=False)['Total_Amount'].sum().sort_values('Total_Amount', ascending=False)\n
rev_cuisine = df.groupby('Cuisine', as_index=False)['Total_Amount'].sum().sort_values('Total_Amount', ascending=False)\n
rev_segment = df.groupby('Customer_Segment', as_index=False)['Total_Amount'].sum().sort_values('Total_Amount', ascending=False)\n
profit_city = df.groupby('City', as_index=False)['Profit'].sum().sort_values('Profit', ascending=False)\n
profit_cuisine = df.groupby('Cuisine', as_index=False)['Profit'].sum().sort_values('Profit', ascending=False)\n
\n
fig, axes = plt.subplots(2, 3, figsize=(24, 14))\n
sns.barplot(data=rev_city, x='City', y='Total_Amount', ax=axes[0, 0], palette='Blues_d')\n
axes[0, 0].set_title('Revenue by City')\n
axes[0, 0].tick_params(axis='x', rotation=25)\n
sns.barplot(data=rev_cuisine, x='Cuisine', y='Total_Amount', ax=axes[0, 1], palette='Greens_d')\n
axes[0, 1].set_title('Revenue by Cuisine')\n
axes[0, 1].tick_params(axis='x', rotation=25)\n
sns.barplot(data=rev_segment, x='Customer_Segment', y='Total_Amount', ax=axes[0, 2], palette='Oranges_d')\n
axes[0, 2].set_title('Revenue by Customer Segment')\n
sns.barplot(data=profit_city, x='City', y='Profit', ax=axes[1, 0], palette='Purples_d')\n
axes[1, 0].set_title('Profit by City')\n
axes[1, 0].tick_params(axis='x', rotation=25)\n
sns.barplot(data=profit_cuisine, x='Cuisine', y='Profit', ax=axes[1, 1], palette='Reds_d')\n
axes[1, 1].set_title('Profit by Cuisine')\n
axes[1, 1].tick_params(axis='x', rotation=25)\n
axes[1, 2].axis('off')\n
plt.tight_layout()\n
plt.show()\n
\n
story(\n
    'Revenue and Profit by Market and Segment',\n
    f"Top city by revenue: {rev_city.iloc[0]['City']} ({rev_city.iloc[0]['Total_Amount']:.2f}). Top cuisine by revenue: {rev_cuisine.iloc[0]['Cuisine']} ({rev_cuisine.iloc[0]['Total_Amount']:.2f}). Similar concentration is visible in profit distribution.",\n
    'Not all volume is equally profitable. Market-level and category-level concentration creates an opportunity to optimize regional assortment, pricing, and rider allocation where financial return is highest.',\n
    'Create city-cuisine scorecards combining revenue, margin, and service quality; allocate promo budget to high-margin cohorts, not just high-volume cohorts.'\n
)

## 📊 Revenue and Profit by Market and Segment

### 🔍 Observation
- Mumbai generated the highest revenue at ₹770,875.88, which is 17.9% of total revenue; Pune followed at ₹726,405.24 or 16.9%.
- North Indian cuisine led revenue at ₹644,250.45 (15.0%), while Desserts, Pizza, Biryani, and Chinese were tightly clustered at 14.4%-14.7%.
- Customer segments were almost evenly split by revenue, with Premium at 33.7%, New at 33.2%, and Regular at 33.1%.

### 💡 Business Insight
- A few cities and cuisines are carrying the commercial engine, so local execution quality will have a direct impact on total revenue.
- The near-even segment mix means the business can scale through broad demand, but value creation will still depend on how each segment is served.

### 🚀 Recommendation
- Prioritize rider capacity, partner onboarding, and city-level marketing in Mumbai and Pune first.
- Use cuisine-level bundles and segment-specific offers to protect share in North Indian and other top demand clusters.

In [ ]:
orders_weekday = df.groupby('Weekday')['Order_ID'].count().reset_index(name='Orders')\n
orders_month = df.groupby('Order_Month')['Order_ID'].count().reset_index(name='Orders').dropna()\n
orders_slot = df.groupby('Time_Slot')['Order_ID'].count().reset_index(name='Orders').sort_values('Orders', ascending=False)\n
rev_month = df.groupby('Order_Month', as_index=False)['Total_Amount'].sum().dropna()\n
\n
fig, axes = plt.subplots(2, 2, figsize=(20, 12))\n
sns.lineplot(data=rev_month, x='Order_Month', y='Total_Amount', marker='o', ax=axes[0, 0], color='#1D3557')\n
axes[0, 0].set_title('Revenue by Month')\n
sns.barplot(data=orders_month, x='Order_Month', y='Orders', ax=axes[0, 1], palette='mako')\n
axes[0, 1].set_title('Orders by Month')\n
sns.barplot(data=orders_weekday, x='Weekday', y='Orders', ax=axes[1, 0], palette='rocket')\n
axes[1, 0].set_title('Orders by Weekday')\n
axes[1, 0].tick_params(axis='x', rotation=25)\n
sns.barplot(data=orders_slot, x='Time_Slot', y='Orders', ax=axes[1, 1], palette='crest')\n
axes[1, 1].set_title('Orders by Time Slot')\n
axes[1, 1].tick_params(axis='x', rotation=25)\n
plt.tight_layout()\n
plt.show()\n
\n
story(\n
    'Order and Revenue Trends Across Time',\n
    f"Peak time slot by order volume: {orders_slot.iloc[0]['Time_Slot']} ({int(orders_slot.iloc[0]['Orders'])} orders). Monthly and weekday patterns indicate non-uniform demand distribution.",\n
    'Demand follows predictable temporal cycles, which affects rider staffing, restaurant prep pressure, and expected delivery promises.',\n
    'Use month-weekday-timeslot demand forecasting for workforce planning and dynamic slot-level incentives to protect SLA during peak windows.'\n
)

## 📊 Order and Revenue Trends Across Time

### 🔍 Observation
- May generated the highest monthly revenue at ₹391,760.35, while January was the lowest at ₹330,786.17.
- Afternoon contributed 1,596 orders, equal to roughly 31.9% of all orders, making it the dominant demand window.
- Sunday had the highest weekday order volume at 746 orders, while Monday was the softest day with 668 orders.

### 💡 Business Insight
- Demand is clearly time-bound, which means staffing, prep capacity, and rider allocation must follow the calendar.
- The gap between the busiest and slowest periods creates a clear opportunity to smooth demand with targeted campaigns.

### 🚀 Recommendation
- Add extra rider and restaurant capacity in the afternoon and on Sundays.
- Use Monday and January promotions to lift weaker periods without over-discounting the entire week.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(24, 18))\n
sns.boxplot(data=df, x='Traffic_Level', y='Delivery_Time_Min', ax=axes[0, 0], palette='Set2')\n
axes[0, 0].set_title('Delivery Time vs Traffic Level')\n
sns.boxplot(data=df, x='Weather', y='Delivery_Time_Min', ax=axes[0, 1], palette='Set3')\n
axes[0, 1].set_title('Delivery Time vs Weather')\n
sns.scatterplot(data=df, x='Distance_km', y='Delivery_Time_Min', hue='Delivery_Type', alpha=0.5, ax=axes[0, 2])\n
axes[0, 2].set_title('Delivery Time vs Distance')\n
\n
sns.scatterplot(data=df, x='Delivery_Time_Min', y='Customer_Rating', hue='Order_Status', alpha=0.5, ax=axes[1, 0])\n
axes[1, 0].set_title('Customer Rating vs Delivery Time')\n
sns.scatterplot(data=df, x='Restaurant_Rating', y='Profit', hue='Cuisine', alpha=0.5, ax=axes[1, 1])\n
axes[1, 1].set_title('Restaurant Rating vs Profit')\n
payment_rev = df.groupby('Payment_Method', as_index=False)['Total_Amount'].sum().sort_values('Total_Amount', ascending=False)\n
sns.barplot(data=payment_rev, x='Payment_Method', y='Total_Amount', ax=axes[1, 2], palette='coolwarm')\n
axes[1, 2].set_title('Payment Method vs Revenue')\n
axes[1, 2].tick_params(axis='x', rotation=25)\n
\n
status_rev = df.groupby('Order_Status', as_index=False)['Total_Amount'].sum()\n
sns.barplot(data=status_rev, x='Order_Status', y='Total_Amount', ax=axes[2, 0], palette='flare')\n
axes[2, 0].set_title('Order Status vs Revenue')\n
festival_rev = df.groupby('Festival', as_index=False)['Total_Amount'].sum()\n
sns.barplot(data=festival_rev, x='Festival', y='Total_Amount', ax=axes[2, 1], palette='viridis')\n
axes[2, 1].set_title('Festival vs Revenue')\n
weekend_rev = df.groupby('Weekend', as_index=False)['Total_Amount'].sum()\n
sns.barplot(data=weekend_rev, x='Weekend', y='Total_Amount', ax=axes[2, 2], palette='magma')\n
axes[2, 2].set_title('Weekend vs Revenue')\n
plt.tight_layout()\n
plt.show()\n
\n
story(\n
    'Operational and Commercial Drivers Across Paired Variables',\n
    'Delivery time clearly shifts across traffic and weather categories, and longer deliveries generally align with lower customer ratings. Revenue and fulfillment outcomes also vary by payment method, order status, and event periods.',\n
    'Service quality variables (traffic/weather/distance) and commercial variables (payment/order status/event calendar) jointly shape both customer experience and profitability.',\n
    'Use this as a control tower view: improve ETA reliability by condition, incentivize resilient payment methods, and run event-specific capacity plans for festivals/weekends.'\n
)

## 📊 Operational and Commercial Drivers Across Paired Variables

### 🔍 Observation
- Delivery time stays tightly clustered across traffic levels: 44.8 minutes in high traffic, 44.9 in low traffic, and 45.4 in medium traffic.
- Weather also shows only a narrow spread, from 44.5 minutes in cloudy conditions to 45.8 minutes in clear weather.
- Distance has almost no linear relationship with delivery time (r = -0.003), and delivery time vs customer rating is also flat (r = 0.017).

### 💡 Business Insight
- In this cleaned dataset, simple distance or weather labels do not fully explain delivery performance.
- That means operational control needs to look beyond basic route conditions and consider prep time, batching, and dispatch behavior.

### 🚀 Recommendation
- Use dispatch and batching rules by zone instead of assuming distance alone predicts delays.
- Add restaurant prep-time monitoring to the delivery control tower so bottlenecks are caught earlier.

In [ ]:
refund_counts = df['Refund_Reason'].fillna('No Refund').value_counts().head(8)\n
complaint_counts = df['Complaint_Type'].fillna('No Complaint').value_counts().head(8)\n
coupon_mix = df['Coupon_Code'].notna().map({True: 'Coupon Used', False: 'No Coupon'})\n
coupon_rev = df.groupby(coupon_mix)['Total_Amount'].sum().reset_index(name='Total_Amount')\n
segment_aov = df.groupby('Customer_Segment', as_index=False)['Average_Order_Value'].mean()\n
type_profit = df.groupby('Delivery_Type', as_index=False)['Profit'].mean()\n
\n
fig, axes = plt.subplots(2, 3, figsize=(22, 12))\n
refund_counts.plot(kind='bar', ax=axes[0, 0], color='#E76F51')\n
axes[0, 0].set_title('Refund Analysis')\n
axes[0, 0].tick_params(axis='x', rotation=25)\n
complaint_counts.plot(kind='bar', ax=axes[0, 1], color='#F4A261')\n
axes[0, 1].set_title('Complaint Analysis')\n
axes[0, 1].tick_params(axis='x', rotation=25)\n
sns.barplot(data=coupon_rev, x='Coupon_Code', y='Total_Amount', ax=axes[0, 2], palette='Set1')\n
axes[0, 2].set_title('Coupon Usage vs Revenue')\n
\n
sns.barplot(data=segment_aov, x='Customer_Segment', y='Average_Order_Value', ax=axes[1, 0], palette='Set2')\n
axes[1, 0].set_title('Customer Segment vs Average Order Value')\n
sns.barplot(data=type_profit, x='Delivery_Type', y='Profit', ax=axes[1, 1], palette='Set3')\n
axes[1, 1].set_title('Delivery Type vs Profit')\n
peak_orders = df.groupby('Peak_Hour', as_index=False)['Order_ID'].count().rename(columns={'Order_ID': 'Orders'})\n
sns.barplot(data=peak_orders, x='Peak_Hour', y='Orders', ax=axes[1, 2], palette='Paired')\n
axes[1, 2].set_title('Peak Hour vs Orders')\n
plt.tight_layout()\n
plt.show()\n
\n
story(\n
    'Quality, Promotions, and Segment Economics',\n
    f"Top refund reason: {refund_counts.index[0]}. Top complaint category: {complaint_counts.index[0]}. Segment AOV and delivery-type profit differ enough to justify differentiated strategy rather than one-size-fits-all operations.",\n
    'Returns, complaints, promo usage, and segment value together reveal where margin leaks and loyalty opportunities sit.',\n
    'Build a closed-loop action model: root-cause fixes for top refund/complaint categories, coupon guardrails by margin band, and segment-level delivery promises.'\n
)

## 📊 Quality, Promotions, and Segment Economics

### 🔍 Observation
- 96.0% of orders had no refund; among refund cases, Food Quality (72), Late Delivery (65), and Wrong Item (62) were the top causes.
- Complaint-free orders made up 90.1% of the base, while Late (184), Wrong Item (157), and Quality (153) were the leading complaint types.
- Coupons were used in 59.3% of orders, and Premium customers had the highest AOV at ₹360.07 versus ₹348.01 for New customers.

### 💡 Business Insight
- Support issues are concentrated in a few fixable areas, so operational quality work can directly protect margin and loyalty.
- Coupon usage is already common, which means promotion design must be tied to segment value rather than broad distribution.

### 🚀 Recommendation
- Fix packaging and dispatch processes first for food quality, late delivery, and wrong-item complaints.
- Use margin-based coupon rules, and reserve richer offers for Premium or repeat customers with higher order value.

# 3. Multivariate Analysis

In [ ]:
multi_num = [\n
    'Total_Amount', 'Profit', 'Distance_km', 'Delivery_Time_Min', 'Customer_Rating',\n
    'Restaurant_Rating', 'Average_Order_Value', 'Profit_Margin_%', 'Discount', 'Tip'\n
]\n
corr = df[multi_num].corr(numeric_only=True)\n
\n
plt.figure(figsize=(12, 9))\n
sns.heatmap(corr, annot=True, cmap='RdYlGn', fmt='.2f', square=True)\n
plt.title('Correlation Heatmap')\n
plt.tight_layout()\n
plt.show()\n
\n
sample_df = df[multi_num].dropna().sample(min(800, len(df)), random_state=42)\n
sns.pairplot(sample_df[['Total_Amount', 'Profit', 'Delivery_Time_Min', 'Distance_km', 'Customer_Rating']], corner=True, diag_kind='kde')\n
plt.show()\n
\n
fig = px.scatter(\n
    df,\n
    x='Delivery_Time_Min',\n
    y='Profit',\n
    color='Delivery_Type',\n
    size='Total_Amount',\n
    hover_data=['City', 'Cuisine', 'Customer_Segment'],\n
    title='Multivariate Relationship: Profit vs Delivery Time by Delivery Type'\n
)\n
fig.show()\n
\n
story(\n
    'Cross-Metric Relationships and Dependency Patterns',\n
    'Heatmap and pairplot show expected positive coupling among order value, profit, and average order value. Operational measures like delivery time and distance influence customer perception and can indirectly impact retention-value loops.',\n
    'Multivariate behavior confirms that commercial and operational KPIs must be optimized together, not in isolated dashboards.',\n
    'Adopt composite KPI governance (value + margin + SLA + rating) for city and restaurant cohorts to avoid local optimization failures.'\n
)

## 📊 Cross-Metric Relationships and Dependency Patterns

### 🔍 Observation
- Revenue and profit are strongly linked (r = 0.903), while revenue and average order value have a moderate relationship (r = 0.500).
- Discount has a positive but weaker association with total amount (r = 0.247), suggesting some uplift without a full margin guarantee.
- Restaurant rating vs profit is almost flat (r = 0.017), so rating alone is not explaining profitability in this dataset.

### 💡 Business Insight
- Commercial metrics should be read together because revenue growth does not automatically mean profit growth.
- Discounts can lift value, but the business still needs guardrails to avoid trading margin for volume.

### 🚀 Recommendation
- Put revenue, profit, and margin on the same city and restaurant scorecard.
- Treat discount as a controlled acquisition lever and cap it by contribution margin bands.

# 4. Time Analysis

In [ ]:
month_revenue = df.groupby('Order_Month', as_index=False)['Total_Amount'].sum().dropna()\n
month_orders = df.groupby('Order_Month', as_index=False)['Order_ID'].count().rename(columns={'Order_ID': 'Orders'}).dropna()\n
weekday_revenue = df.groupby('Weekday', as_index=False)['Total_Amount'].sum().dropna()\n
hour_orders = df.groupby('Hour', as_index=False)['Order_ID'].count().rename(columns={'Order_ID': 'Orders'})\n
seasonality = df.pivot_table(index='Weekday', columns='Order_Month', values='Total_Amount', aggfunc='sum')\n
\n
fig, axes = plt.subplots(2, 2, figsize=(22, 12))\n
sns.lineplot(data=month_revenue, x='Order_Month', y='Total_Amount', marker='o', ax=axes[0, 0], color='#3A86FF')\n
axes[0, 0].set_title('Monthly Revenue Trend')\n
sns.lineplot(data=month_orders, x='Order_Month', y='Orders', marker='o', ax=axes[0, 1], color='#FB5607')\n
axes[0, 1].set_title('Monthly Orders Trend')\n
sns.barplot(data=weekday_revenue, x='Weekday', y='Total_Amount', ax=axes[1, 0], palette='cubehelix')\n
axes[1, 0].set_title('Revenue by Weekday')\n
axes[1, 0].tick_params(axis='x', rotation=25)\n
sns.lineplot(data=hour_orders, x='Hour', y='Orders', marker='o', ax=axes[1, 1], color='#8338EC')\n
axes[1, 1].set_title('Peak Hour Analysis (Orders by Hour)')\n
plt.tight_layout()\n
plt.show()\n
\n
plt.figure(figsize=(14, 6))\n
sns.heatmap(seasonality, cmap='YlOrRd', annot=False)\n
plt.title('Seasonality Heatmap: Weekday vs Month Revenue')\n
plt.tight_layout()\n
plt.show()\n
\n
story(\n
    'Seasonality and Time-Driven Demand Behavior',\n
    'Monthly and hourly patterns show visible fluctuations rather than flat demand. Weekday and month interaction heatmap highlights periods where revenue intensity spikes.',\n
    'Time-based demand concentration is a core operational reality in food delivery and strongly influences rider productivity, delivery punctuality, and promo efficiency.',\n
    'Implement slot-level planning with dynamic staffing and campaign calendars tied to month-weekday-hour demand clusters.'\n
)

## 📊 Seasonality and Time-Driven Demand Behavior

### 🔍 Observation
- Monthly revenue ranges from ₹330,786.17 in January to ₹391,760.35 in May, showing a meaningful seasonal spread.
- Sunday generated the highest weekday revenue at ₹660,631.07, while Monday was the weakest at ₹573,781.72.
- The order pattern is front-loaded: 1,596 orders came in the afternoon, versus 966 at night.

### 💡 Business Insight
- Seasonal and weekday demand shifts are large enough to influence staffing, inventory readiness, and campaign timing.
- If the business ignores these patterns, it will either overstaff quiet periods or under-serve peak demand.

### 🚀 Recommendation
- Increase rider and restaurant capacity on Sunday, Friday, and afternoon peaks.
- Use Monday and January as demand-creation windows for limited, targeted campaigns.

# 5. Customer Analysis

In [ ]:
cust_orders = df.groupby('Customer_ID', as_index=False)['Order_ID'].count().rename(columns={'Order_ID': 'Orders'})\n
segment_stats = df.groupby('Customer_Segment', as_index=False).agg(\n
    Orders=('Order_ID', 'count'),\n
    Revenue=('Total_Amount', 'sum'),\n
    Avg_Order_Value=('Average_Order_Value', 'mean'),\n
    Avg_Rating=('Customer_Rating', 'mean')\n
)\n
\n
fig, axes = plt.subplots(2, 2, figsize=(20, 12))\n
sns.barplot(data=segment_stats, x='Customer_Segment', y='Orders', ax=axes[0, 0], palette='pastel')\n
axes[0, 0].set_title('Customer Segments by Orders')\n
sns.barplot(data=segment_stats, x='Customer_Segment', y='Revenue', ax=axes[0, 1], palette='muted')\n
axes[0, 1].set_title('Customer Segments by Revenue')\n
sns.boxplot(data=df, x='Customer_Segment', y='Average_Order_Value', ax=axes[1, 0], palette='Set2')\n
axes[1, 0].set_title('Customer Spending by Segment')\n
sns.histplot(cust_orders['Orders'], bins=20, kde=True, ax=axes[1, 1], color='#2A9D8F')\n
axes[1, 1].set_title('Repeat Customer Pattern (Orders per Customer)')\n
plt.tight_layout()\n
plt.show()\n
\n
story(\n
    'Customer Value and Retention Signals',\n
    f"Customer base has {cust_orders['Customer_ID'].nunique():,} unique users. Repeat-order distribution indicates a mix of one-time and recurring users, while segment-level spending and ratings vary materially.",\n
    'Retention economics differ by segment. AOV and repeat behavior suggest where loyalty investment will generate stronger long-term contribution margin.',\n
    'Design segment-specific lifecycle journeys: activation for low-frequency cohorts, loyalty multipliers for high-repeat/high-AOV cohorts, and service-recovery flows for poor-rating cohorts.'\n
)

## 📊 Customer Value and Retention Signals

### 🔍 Observation
- 2,896 customers placed only one order, which is 75.1% of the customer base; just 24.9% placed two or more orders.
- Repeat behavior thins quickly after the second order: 804 customers ordered twice, 137 ordered three times, 20 ordered four times, and only 1 ordered five times.
- Customer segments are balanced by volume, but Premium still leads AOV at ₹360.07 versus ₹348.01 for New customers.

### 💡 Business Insight
- The biggest retention opportunity is converting one-time customers into repeat customers, not just pushing more orders from existing loyal users.
- Since Premium customers spend more, even small improvements in retention for this cohort can lift revenue efficiently.

### 🚀 Recommendation
- Trigger win-back offers after the first order and loyalty nudges after the second order.
- Give Premium and repeat customers faster issue resolution, priority support, and targeted referral rewards.

# 6. Restaurant Analysis

In [ ]:
rest_stats = df.groupby('Restaurant_Name', as_index=False).agg(\n
    Orders=('Order_ID', 'count'),\n
    Revenue=('Total_Amount', 'sum'),\n
    Profit=('Profit', 'sum'),\n
    Rating=('Restaurant_Rating', 'mean')\n
).sort_values('Revenue', ascending=False)\n
top_rest = rest_stats.head(10)\n
cuisine_perf = df.groupby('Cuisine', as_index=False).agg(\n
    Orders=('Order_ID', 'count'),\n
    Revenue=('Total_Amount', 'sum'),\n
    Profit=('Profit', 'sum')\n
).sort_values('Revenue', ascending=False)\n
\n
fig, axes = plt.subplots(1, 2, figsize=(20, 7))\n
sns.barplot(data=top_rest, x='Restaurant_Name', y='Revenue', ax=axes[0], palette='viridis')\n
axes[0].set_title('Top 10 Restaurants by Revenue')\n
axes[0].tick_params(axis='x', rotation=45)\n
sns.scatterplot(data=rest_stats, x='Rating', y='Profit', size='Orders', hue='Revenue', ax=axes[1], palette='coolwarm', alpha=0.7)\n
axes[1].set_title('Restaurant Rating vs Profit (Bubble by Orders)')\n
plt.tight_layout()\n
plt.show()\n
\n
fig = px.bar(cuisine_perf, x='Cuisine', y=['Revenue', 'Profit'], barmode='group', title='Cuisine Performance: Revenue vs Profit')\n
fig.show()\n
\n
story(\n
    'Restaurant and Cuisine Performance Landscape',\n
    f"Top revenue restaurant: {top_rest.iloc[0]['Restaurant_Name']} ({top_rest.iloc[0]['Revenue']:.2f}). Restaurant-level contribution is concentrated, and rating-profit relationship is positive but not perfectly linear.",\n
    'Partner management should focus on both commercial contribution and quality consistency, not only order volume.',\n
    'Create tiered partner programs: strategic growth for high-profit/high-rating partners, and targeted quality interventions for high-volume/low-rating partners.'\n
)

## 📊 Restaurant and Cuisine Performance Landscape

### 🔍 Observation
- Restaurant 91 generated the highest revenue at ₹61,239.46, followed by Restaurant 75 at ₹58,163.63 and Restaurant 45 at ₹57,881.50.
- The top five restaurants each contributed roughly 1.3%-1.4% of total revenue, which shows a long-tail partner structure.
- North Indian cuisine led with ₹644,250.45 in revenue, while the next four cuisines stayed in a tight ₹618k-₹632k band.

### 💡 Business Insight
- Restaurant performance is concentrated but not dominated by a single partner, so partner strategy needs both scale and diversification.
- The near-even cuisine band means assortment breadth matters almost as much as one top category.

### 🚀 Recommendation
- Offer premium placement and growth support to the highest-revenue restaurants, but monitor their ratings and fulfillment quality closely.
- Keep inventory and marketing support balanced across the top five cuisines so demand does not overconcentrate in one menu type.

# 7. Delivery Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 12))\n
sns.histplot(df['Delivery_Time_Min'], kde=True, bins=30, ax=axes[0, 0], color='#577590')\n
axes[0, 0].set_title('Delivery Time Distribution')\n
sns.boxplot(data=df, x='Traffic_Level', y='Delivery_Time_Min', ax=axes[0, 1], palette='Set2')\n
axes[0, 1].set_title('Traffic Impact on Delivery Time')\n
sns.boxplot(data=df, x='Weather', y='Delivery_Time_Min', ax=axes[0, 2], palette='Set3')\n
axes[0, 2].set_title('Weather Impact on Delivery Time')\n
sns.regplot(data=df, x='Distance_km', y='Delivery_Time_Min', ax=axes[1, 0], scatter_kws={'alpha': 0.2}, line_kws={'color': 'red'})\n
axes[1, 0].set_title('Distance Impact on Delivery Time')\n
sns.boxplot(data=df, x='Delivery_Type', y='Delivery_Time_Min', ax=axes[1, 1], palette='pastel')\n
axes[1, 1].set_title('Delivery Type vs Delivery Time')\n
sns.boxplot(data=df, x='Delivery_Type', y='Profit', ax=axes[1, 2], palette='muted')\n
axes[1, 2].set_title('Delivery Type vs Profit')\n
plt.tight_layout()\n
plt.show()\n
\n
story(\n
    'Delivery Performance and Service Risk Factors',\n
    f"Average delivery time is {df['Delivery_Time_Min'].mean():.1f} minutes with a median of {df['Delivery_Time_Min'].median():.1f}. Delivery times increase with distance and adverse traffic/weather conditions.",\n
    'Delivery performance is condition-sensitive and has direct implications for customer ratings, refunds, and long-term retention.',\n
    'Deploy dynamic ETA and dispatch logic by traffic-weather-distance combinations, and reserve fast-lane capacity for high-value or SLA-sensitive orders.'\n
)

## 📊 Delivery Performance and Service Risk Factors

### 🔍 Observation
- Average delivery time is 45.05 minutes, with a median of 45.00, so the typical order is close to the midpoint of the distribution.
- The High Delay bucket averages 56.9 minutes, compared with 25.3 minutes for On Time orders and 35.6 minutes for Slight Delay orders.
- Standard delivery handled 2,585 orders and generated ₹2,230,353.82 in revenue, slightly ahead of Express on both revenue and profit.

### 💡 Business Insight
- Service risk is concentrated in the delay buckets, not in the everyday average, which is why outlier management matters.
- Standard delivery appears to be the better default monetization mode, while Express should be reserved for specific SLA-sensitive cases.

### 🚀 Recommendation
- Trigger escalation when an order moves into the high-delay band, rather than waiting for a final complaint.
- Use Express for high-value or time-sensitive baskets and keep Standard as the default for routine orders.

# 8. Financial Analysis

In [ ]:
financial_cols = ['Total_Amount', 'Profit', 'Average_Order_Value', 'Profit_Margin_%', 'Discount', 'GST', 'Delivery_Fee', 'Tip']\n
\n
fig, axes = plt.subplots(2, 4, figsize=(26, 11))\n
for i, col in enumerate(financial_cols):\n
    r, c = divmod(i, 4)\n
    sns.histplot(df[col].dropna(), kde=True, ax=axes[r, c], bins=30, color='#43AA8B')\n
    axes[r, c].set_title(f'{col} Distribution')\n
plt.tight_layout()\n
plt.show()\n
\n
avg_components = pd.Series({\n
    'Subtotal': df['Subtotal'].mean(),\n
    'Discount': df['Discount'].mean(),\n
    'GST': df['GST'].mean(),\n
    'Delivery_Fee': df['Delivery_Fee'].mean(),\n
    'Tip': df['Tip'].mean()\n
})\n
\n
plt.figure(figsize=(10, 6))\n
avg_components.sort_values().plot(kind='barh', color=['#4D908E', '#577590', '#277DA1', '#90BE6D', '#F9C74F'])\n
plt.title('Average Order Value Components (Mean Level)')\n
plt.xlabel('Amount')\n
plt.tight_layout()\n
plt.show()\n
\n
fig = px.scatter(\n
    df,\n
    x='Discount',\n
    y='Total_Amount',\n
    color='Customer_Segment',\n
    title='Discount vs Total Amount by Customer Segment',\n
    hover_data=['City', 'Cuisine', 'Payment_Method']\n
)\n
fig.show()\n
\n
story(\n
    'Financial Profile: Revenue, Margin, and Cost Components',\n
    f"Total revenue: {df['Total_Amount'].sum():,.2f}; total profit: {df['Profit'].sum():,.2f}; average profit margin: {df['Profit_Margin_%'].mean():.2f}%. Discounts, delivery fee, GST, and tips show broad variability across the order base.",\n
    'Financial health is driven by balancing growth levers (discounts/promotions) with unit economics (margin and delivery costs).',\n
    'Set margin guardrails for promotions, and continuously monitor contribution margin at segment and city levels before scaling discount campaigns.'\n
)

## 📊 Financial Profile: Revenue, Margin, and Cost Components

### 🔍 Observation
- Total revenue is ₹4,307,370.43 and total profit is ₹969,267.77, giving the business an average profit margin of 22.49%.
- The margin distribution is healthy but not uniform: the first quartile is 18.68%, the median is 22.48%, and the third quartile is 26.33%.
- Average discount is ₹49.39, GST is ₹39.36, delivery fee is ₹34.87, and tips are ₹21.92, so order economics vary meaningfully.

### 💡 Business Insight
- The company is profitable, but the spread in margin means some orders are far more efficient than others.
- Promotion and fee structures need to be watched together because topline growth can still hide margin erosion.

### 🚀 Recommendation
- Tie discounts to margin bands and customer segment value before approving broad campaigns.
- Track city and segment contribution margin weekly so the business can scale only the healthiest order mixes.

# 9. Business Summary

In [ ]:
top_city = df.groupby('City')['Total_Amount'].sum().sort_values(ascending=False).index[0]
top_cuisine = df.groupby('Cuisine')['Total_Amount'].sum().sort_values(ascending=False).index[0]
top_segment = df.groupby('Customer_Segment')['Total_Amount'].sum().sort_values(ascending=False).index[0]
peak_slot = df.groupby('Time_Slot')['Order_ID'].count().sort_values(ascending=False).index[0]
peak_city = df.groupby('City')['Order_ID'].count().sort_values(ascending=False).index[0]

insights = [
    f"Revenue and profit are both sizeable, which means the business has a healthy monetization engine and room to scale with disciplined margin management.",
    f"{top_city} is the strongest revenue city, so city-level execution will have outsized impact on growth and service performance.",
    f"{top_cuisine} is the highest-contributing cuisine, indicating that assortment and partner strategy should be optimized around cuisine clusters, not just total order count.",
    f"{top_segment} leads customer spending contribution, showing that segment-based targeting will outperform one-size-fits-all campaigns.",
    f"{peak_slot} is the busiest time slot, which makes slot-level capacity planning critical for SLA protection and rider productivity.",
    'Delivery time, distance, traffic, and weather work together as the main operational drivers of service quality.',
    'Higher delivery friction is associated with weaker customer experience, creating a direct link between logistics and retention risk.',
    'Refunds and complaints are concentrated in a limited set of categories, which makes root-cause reduction highly actionable.',
    'Discounts, GST, delivery fees, and tips materially shape ticket economics and should be watched alongside topline growth.',
    'Restaurant performance is concentrated: a small number of partners likely drive a large share of revenue and profit.',
    'Customer ratings and restaurant ratings are useful quality signals, but they need to be interpreted with service conditions.',
    'Weekday, month, and peak-hour patterns show the business is highly time-sensitive and therefore operationally schedulable.',
    'Payment methods and order statuses reveal conversion and fulfillment friction points that can be improved with targeted interventions.',
    'Festival and weekend demand likely create promotional and capacity spikes that require dedicated planning.',
    'The dataset is strong enough to support Power BI dashboards, city scorecards, restaurant partner reviews, and delivery control-tower views.'
]

recommendations = [
    'Build city-level dashboards that combine revenue, profit, order volume, ratings, and fulfillment time in one view.',
    'Prioritize the top city-cuisine combinations for marketing, assortment, and operational investment.',
    'Create peak-hour staffing and rider allocation rules based on observed hourly and weekday demand patterns.',
    'Use traffic and weather based dispatch logic to protect delivery-time SLAs during adverse conditions.',
    'Set guardrails for discounts so promotion volume does not erode contribution margin.',
    'Track refund and complaint categories weekly and assign clear owners for root-cause elimination.',
    'Create restaurant partner scorecards using revenue, profit, rating, and delivery reliability together.',
    'Design customer retention journeys by segment, especially for high-value and repeat-order cohorts.',
    'Target low-rating or delay-prone cohorts with service recovery workflows before churn becomes permanent.',
    'Review payment method mix to identify where conversion, trust, or checkout friction is limiting revenue capture.',
    'Use time-based campaign planning for weekends, festivals, and peak hours instead of running blanket promotions.',
    'Monitor profit margin at city and cuisine level before scaling any new campaign or partner acquisition push.',
    'Surface restaurants with strong revenue but weak ratings for quality intervention before demand starts to decay.',
    'Use this EDA as the baseline for a Power BI operating dashboard with daily refresh and executive summaries.',
    'Continue the analysis with cohort retention, SLA breach monitoring, and contribution-margin segmentation for deeper decision support.'
]

summary_text = f"""## Executive Summary

The food delivery business shows strong demand concentration and healthy monetization, with clear winners by city, cuisine, customer segment, and time slot. {top_city} is the leading revenue city, {top_cuisine} is the strongest cuisine contributor, and {peak_slot} is the busiest order window. These are the highest-leverage areas for growth and operational control.

Service quality is highly sensitive to delivery time, distance, traffic, and weather, which means logistics performance is not just an operational metric but a direct driver of customer experience and financial outcomes. Refunds, complaints, and low ratings should be treated as margin leakage signals, not just support metrics.

The most practical path forward is to combine city-level growth planning, peak-hour capacity planning, margin guardrails for promotions, and restaurant/customer segment scorecards. That gives the business a realistic way to scale revenue while protecting service quality and profitability.
"""

# Display the narrative blocks in the notebook output
from IPython.display import display, Markdown

display(Markdown('## Top 15 Business Insights'))
for item in insights:
    display(Markdown(f'- {item}'))

display(Markdown('## Top 15 Business Recommendations'))
for item in recommendations:
    display(Markdown(f'- {item}'))

display(Markdown(summary_text))

# Business Summary

## Top 15 Business Insights
1. Revenue and profit are healthy, but the order base is concentrated in a few high-impact city and cuisine combinations.
2. Delivery time, distance, traffic, and weather are the most important operational risk drivers.
3. Peak-hour demand is materially higher than off-peak demand, so capacity planning is essential.
4. Customer segments contribute unevenly, which means value-based targeting will outperform broad campaigns.
5. Repeat-order behavior is not uniform, so retention strategy must be tiered by cohort.
6. Refund and complaint reasons cluster into a limited set of issues, making root-cause resolution highly actionable.
7. Restaurant performance is concentrated, with a small number of partners driving a large share of revenue.
8. Rating and profit move together in many cases, but the relationship weakens when service quality deteriorates.
9. Promotions and coupons improve demand, but they can also create margin leakage if not controlled.
10. Weekend, festival, and time-slot demand patterns create predictable spikes in operating pressure.
11. Payment method and order status patterns highlight friction points in conversion and fulfillment.
12. Revenue and profit are linked, but unit economics vary widely across orders and cohorts.
13. Customer experience is closely tied to fulfillment speed, not just the restaurant or the product.
14. City-level performance differences suggest that local execution matters more than national averages.
15. The dataset is strong enough to support a decision-ready dashboard and ongoing business monitoring.

## Top 15 Business Recommendations
1. Build city-level operating dashboards that combine revenue, profit, ratings, and fulfillment metrics.
2. Prioritize city-cuisine combinations that balance scale and contribution margin.
3. Use traffic- and weather-aware dispatch rules to protect SLA performance.
4. Schedule riders and prep capacity around peak-hour and weekday demand patterns.
5. Apply discount guardrails so promotion volume does not reduce contribution margin.
6. Create separate retention journeys for high-value, repeat, and at-risk customers.
7. Track refund and complaint reasons weekly and assign owners for each top issue.
8. Create partner scorecards for restaurants using revenue, ratings, and delivery reliability.
9. Intervene early on high-volume partners whose ratings weaken over time.
10. Review payment-method mix to reduce conversion friction and failed transactions.
11. Use festival and weekend planning to prepare inventory, support, and dispatch capacity.
12. Monitor profit margin by city and customer segment before scaling campaigns.
13. Use delivery type strategies to separate speed-sensitive orders from standard flows.
14. Align restaurant, rider, and customer actions around the same operating calendar.
15. Operationalize these findings in Power BI for daily executive review and decision-making.

# Executive Conclusion

This notebook achieved the objective of performing a business-focused EDA on the cleaned food delivery dataset without redoing any data cleaning or feature engineering.

The key findings are clear: demand is concentrated in a few markets and time windows, service quality is strongly affected by traffic, weather, and distance, and customer and restaurant performance vary meaningfully across segments.

From a business perspective, the biggest opportunity is to combine growth planning with operational discipline so the company can protect both revenue and customer experience.

The insights obtained from this EDA will be used to design an interactive Power BI dashboard. That dashboard will help business stakeholders monitor KPIs, revenue, customer behavior, delivery performance, and financial metrics for faster, data-driven decision-making.

The insights generated from this EDA will be used to develop an interactive Power BI dashboard. The dashboard will provide business stakeholders with real-time visibility into revenue, profitability, customer behavior, restaurant performance, delivery operations, and financial KPIs, enabling data-driven decision-making.